### Setup

In [0]:
from pyspark.sql.functions import col, row_number
from pyspark.sql.window import Window

print("="*70)
print("GOLD LAYER: Dimensional Model (Star Schema)")
print("="*70)
print("\nBuilding:")
print("  • dim_customers (customer master)")
print("  • dim_products (product master)")
print("  • fact_sales (sales transactions)")

### Build dim_customers (Customer Dimension)

In [0]:
from pyspark.sql.functions import col, substring
from pyspark.sql.types import IntegerType

print(f"\n{'='*70}")
print("Building: dim_customers")
print(f"{'='*70}")

# Read Silver customer tables
cust_info = spark.read.table("`databricks-medallion-lakehouse`.silver.cust_info")
cust_az12 = spark.read.table("`databricks-medallion-lakehouse`.silver.cust_az12")
loc_a101 = spark.read.table("`databricks-medallion-lakehouse`.silver.loc_a101")

# Extract numeric customer_id from ERP strings
# "NASAW00011000" → last 5 digits → "11000"
cust_az12 = cust_az12.withColumn(
    "customer_id_match",
    substring(col("customer_id_erp"), -5, 5).cast(IntegerType())  # ✅ Fixed
)

loc_a101 = loc_a101.withColumn(
    "customer_id_match",
    substring(col("customer_id_erp"), -5, 5).cast(IntegerType())  # ✅ Fixed
)

# Join all 3 customer tables
dim_customers = cust_info \
    .join(cust_az12, cust_info["customer_id"] == cust_az12["customer_id_match"], "left") \
    .join(loc_a101, cust_info["customer_id"] == loc_a101["customer_id_match"], "left") \
    .select(
        cust_info["customer_id"],
        cust_info["customer_key"],
        cust_info["first_name"],
        cust_info["last_name"],
        cust_info["gender"],
        cust_info["marital_status"],
        cust_info["created_date"],
        cust_az12["birthdate"],
        loc_a101["country"]
    )

rows = dim_customers.count()
print(f"  Rows: {rows:,}")
print(f"\n  First 3 rows:")
dim_customers.show(3, truncate=False)

### Write dim_customers to Gold

In [0]:
# Write dim_customers to Gold
gold_table = "`databricks-medallion-lakehouse`.gold.dim_customers"

spark.sql(f"DROP TABLE IF EXISTS {gold_table}")

dim_customers.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable(gold_table)

print(f"✅ Written to Gold: {gold_table}")

### Build dim_products (Product Dimension)

In [0]:
print(f"\n{'='*70}")
print("Building: dim_products")
print(f"{'='*70}")

# Read Silver product tables
prd_info = spark.read.table("`databricks-medallion-lakehouse`.silver.prd_info")
px_cat = spark.read.table("`databricks-medallion-lakehouse`.silver.px_cat_g1v2")

# Join product info with categories
dim_products = prd_info \
    .join(px_cat, prd_info["product_line"] == px_cat["category_id"], "left") \
    .select(
        prd_info["product_id"],
        prd_info["product_key"],
        prd_info["product_name"],
        prd_info["cost"],
        prd_info["product_line"],
        px_cat["category"],
        px_cat["subcategory"],
        px_cat["maintenance_required"],
        prd_info["start_date"],
        prd_info["end_date"]
    )

rows = dim_products.count()
print(f"  Rows: {rows:,}")
print(f"\n  First 3 rows:")
dim_products.show(3, truncate=False)

# Write to Gold
gold_table = "`databricks-medallion-lakehouse`.gold.dim_products"

spark.sql(f"DROP TABLE IF EXISTS {gold_table}")

dim_products.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable(gold_table)

print(f"\n✅ Written to Gold: {gold_table}")

### Build fact_sales (Sales Fact Table)

In [0]:
print(f"\n{'='*70}")
print("Building: fact_sales")
print(f"{'='*70}")

# Read Silver sales and Gold dimensions
sales = spark.read.table("`databricks-medallion-lakehouse`.silver.sales_details")
dim_cust = spark.read.table("`databricks-medallion-lakehouse`.gold.dim_customers")
dim_prod = spark.read.table("`databricks-medallion-lakehouse`.gold.dim_products")

# Join sales with dimensions to get surrogate keys
fact_sales = sales \
    .join(dim_cust, sales["customer_id"] == dim_cust["customer_id"], "left") \
    .join(dim_prod, sales["product_key"] == dim_prod["product_key"], "left") \
    .select(
        sales["order_number"],
        dim_cust["customer_id"].alias("customer_key"),
        dim_prod["product_id"].alias("product_key"),
        sales["order_date"],
        sales["ship_date"],
        sales["due_date"],
        sales["sales_amount"],
        sales["quantity"],
        sales["price"]
    )

rows = fact_sales.count()
print(f"  Rows: {rows:,}")
print(f"\n  First 3 rows:")
fact_sales.show(3, truncate=False)

# Write to Gold
gold_table = "`databricks-medallion-lakehouse`.gold.fact_sales"

spark.sql(f"DROP TABLE IF EXISTS {gold_table}")

fact_sales.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable(gold_table)

print(f"\n✅ Written to Gold: {gold_table}")

### Validation Queries

In [0]:
print("\n" + "="*70)
print("VALIDATION: Gold Layer Tables")
print("="*70)

# Count rows in each table
print(f"\ndim_customers: {spark.read.table('`databricks-medallion-lakehouse`.gold.dim_customers').count():,} rows")
print(f"dim_products:  {spark.read.table('`databricks-medallion-lakehouse`.gold.dim_products').count():,} rows")
print(f"fact_sales:    {spark.read.table('`databricks-medallion-lakehouse`.gold.fact_sales').count():,} rows")

# Sample query: Top 5 customers by sales
print("\nTop 5 Customers by Sales Amount:")
spark.sql("""
    SELECT 
        c.first_name,
        c.last_name,
        SUM(f.sales_amount) as total_sales,
        COUNT(*) as order_count
    FROM `databricks-medallion-lakehouse`.gold.fact_sales f
    JOIN `databricks-medallion-lakehouse`.gold.dim_customers c 
        ON f.customer_key = c.customer_id
    WHERE f.sales_amount > 0
    GROUP BY c.first_name, c.last_name
    ORDER BY total_sales DESC
    LIMIT 5
""").show()

print("\n" + "="*70)
print("✅ GOLD LAYER VALIDATION COMPLETE")
print("="*70)